# Systematic Review Screening Pipeline
## AI and Machine Learning for Bacterial Colony Identification on Agar Plate Images
*Modeled after the clinical medicine systematic review screening workflow (`clinical_medicine_screening.ipynb`)*

---

### Workflow Overview:
1. **Load Screening Instructions & Deduplicated Corpus**: Load the comprehensive criteria from `bacteria_screening_instructions.txt` and the 16,013 deduplicated papers from `deduplicated_papers.csv`.
2. **Partition into 5 Groups**: The 16,013 papers are split into **5 balanced groups** (~3,203 papers each) for staged processing and manual human auditing.
3. **Screen Group 1 First**:
   - Option A: **OpenAI Batch API** (50% cheaper, asynchronous, standard for large systematic reviews).
   - Option B: **Direct API Screening** (with persistent progress checkpointing, live `tqdm` progress, and exponential backoff retry).
4. **Groups 2 to 5 Staged (Protected)**: Code is written for all groups, but **locked** (`RUN_GROUP_X = False`) so they will not run until Group 1 is audited and approved.
5. **Parse, Audit & Merge Results**: Extract JSON decisions (`Include?`, `Exclusion_reason`, `Uncertain`, `Comments`), merge with metadata, and compute agreement metrics (Cohen's Kappa, Sensitivity/Specificity).


## 1. Setup, Environment & Configuration


In [13]:
import os
import sys
import json
import csv
import copy
import time
import getpass
from collections import Counter
from pathlib import Path
from tqdm.auto import tqdm

# Attempt imports for optional data science libraries
try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

# Create project directory structure
os.makedirs("screening_groups", exist_ok=True)
os.makedirs("batch_requests", exist_ok=True)
os.makedirs("screening_results", exist_ok=True)

# API Configuration
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    # Prompt user interactively if running in Jupyter
    # OPENAI_API_KEY = getpass.getpass("Enter your OpenAI API Key (or set OPENAI_API_KEY env var): ")
    pass

# Screening model selection
# Configured for GPT-5.4 Thinking (Deep multi-step reasoning)
# Options for reasoning_effort: "low", "medium", "high", "xhigh"
DEFAULT_MODEL = "gpt-5.4"
DEFAULT_REASONING_EFFORT = "medium"

print(f"Setup complete.")
print(f"Pandas available: {pd is not None}")
print(f"OpenAI client available: {OpenAI is not None}")
print(f"Default screening model: {DEFAULT_MODEL} (Thinking / reasoning_effort: {DEFAULT_REASONING_EFFORT})")


Setup complete.
Pandas available: True
OpenAI client available: True
Default screening model: gpt-4o-mini


## 2. Load Screening Instructions & Deduplicated Papers


In [14]:
# 1. Load screening instructions
INSTRUCTIONS_FILE = "bacteria_screening_instructions.txt"
with open(INSTRUCTIONS_FILE, "r", encoding="utf-8") as f:
    INSTRUCTIONS_RAW = f.read()

# Verify that instructions are valid JSON
try:
    instructions_json = json.loads(INSTRUCTIONS_RAW)
    print("Screening instructions loaded successfully as valid JSON.")
    print("Topic:", instructions_json["instructions"].get("review_topic"))
    print("Period:", instructions_json["instructions"].get("review_period"))
except Exception as e:
    print("Loaded instructions as raw text:", str(e))

# 2. Load deduplicated papers
PAPERS_FILE = "deduplicated_papers.csv"
all_papers = []
with open(PAPERS_FILE, "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    all_papers = list(reader)

total_papers = len(all_papers)
print(f"\nTotal deduplicated papers: {total_papers:,}")

# Database breakdown
db_counts = Counter(p.get("Primary_Database", "Unknown") for p in all_papers)
print("\nDatabase Breakdown:")
for db, cnt in db_counts.most_common():
    print(f"  - {db:15}: {cnt:6,} ({cnt/total_papers*100:.1f}%)")


Screening instructions loaded successfully as valid JSON.
Topic: Artificial Intelligence and Machine Learning for Bacterial Colony Identification on Agar Plate Images
Period: {'start_year': 2010, 'end_year': 2026, 'inclusive': True}

Total deduplicated papers: 16,013

Database Breakdown:
  - Scopus         : 14,941 (93.3%)
  - PubMed         :    709 (4.4%)
  - SpringerLink   :    241 (1.5%)
  - IEEE Xplore    :     98 (0.6%)
  - Embase         :     24 (0.1%)


## 3. Divide Dataset into 5 Groups (For Manual Checking & Staged Screening)
The 16,013 papers are divided into **5 balanced groups**:
- **Group 1**: 3,203 papers (Index 0 – 3,202)
- **Group 2**: 3,203 papers (Index 3,203 – 6,405)
- **Group 3**: 3,203 papers (Index 6,406 – 9,608)
- **Group 4**: 3,202 papers (Index 9,609 – 12,810)
- **Group 5**: 3,202 papers (Index 12,811 – 16,012)

Each group is saved with dedicated columns for manual human verification (`Human_Include`, `Human_Exclusion_Reason`, `Human_Comments`).


In [15]:
n_groups = 5
base_size = total_papers // n_groups
remainder = total_papers % n_groups

group_partitions = {}
start_idx = 0

fieldnames = [
    "Study_ID", "Group", "Group_Index", "Global_Index",
    "Primary_Database", "All_Databases", "Primary_ID", "All_IDs",
    "Title", "Authors", "Year", "Journal", "DOI", "Abstract",
    "Human_Include", "Human_Exclusion_Reason", "Human_Comments",
    "LLM_Include", "LLM_Exclusion_Reason", "LLM_Uncertain", "LLM_Comments"
]

print("Generating 5 Group Partitions...")
for g in range(n_groups):
    size = base_size + (1 if g < remainder else 0)
    end_idx = start_idx + size
    group_num = g + 1
    group_papers = all_papers[start_idx:end_idx]
    
    csv_path = f"screening_groups/group_{group_num}_papers.csv"
    jsonl_path = f"screening_groups/group_{group_num}_papers.jsonl"
    
    with open(csv_path, "w", encoding="utf-8-sig", newline="") as f_csv, \
         open(jsonl_path, "w", encoding="utf-8") as f_jsonl:
        
        writer = csv.DictWriter(f_csv, fieldnames=fieldnames)
        writer.writeheader()
        
        processed_group_records = []
        for g_idx, p in enumerate(group_papers):
            study_id = f"G{group_num}_{g_idx+1:04d}"
            record = {
                "Study_ID": study_id,
                "Group": f"Group {group_num}",
                "Group_Index": g_idx + 1,
                "Global_Index": start_idx + g_idx,
                "Primary_Database": p.get("Primary_Database", ""),
                "All_Databases": p.get("All_Databases", ""),
                "Primary_ID": p.get("Primary_ID", ""),
                "All_IDs": p.get("All_IDs", ""),
                "Title": p.get("Title", "").strip(),
                "Authors": p.get("Authors", "").strip(),
                "Year": p.get("Year", "").strip(),
                "Journal": p.get("Journal", "").strip(),
                "DOI": p.get("DOI", "").strip(),
                "Abstract": p.get("Abstract", "").strip(),
                "Human_Include": "",
                "Human_Exclusion_Reason": "",
                "Human_Comments": "",
                "LLM_Include": "",
                "LLM_Exclusion_Reason": "",
                "LLM_Uncertain": "",
                "LLM_Comments": ""
            }
            writer.writerow(record)
            f_jsonl.write(json.dumps(record, ensure_ascii=False) + "\n")
            processed_group_records.append(record)
            
    group_partitions[group_num] = {
        "count": len(group_papers),
        "start_idx": start_idx,
        "end_idx": end_idx - 1,
        "csv_path": csv_path,
        "jsonl_path": jsonl_path,
        "records": processed_group_records
    }
    
    print(f"  Group {group_num}: {len(group_papers):,} papers (Global {start_idx:,} to {end_idx-1:,}) -> {csv_path}")
    start_idx = end_idx


Generating 5 Group Partitions...
  Group 1: 3,203 papers (Global 0 to 3,202) -> screening_groups/group_1_papers.csv
  Group 2: 3,203 papers (Global 3,203 to 6,405) -> screening_groups/group_2_papers.csv
  Group 3: 3,203 papers (Global 6,406 to 9,608) -> screening_groups/group_3_papers.csv
  Group 4: 3,202 papers (Global 9,609 to 12,810) -> screening_groups/group_4_papers.csv
  Group 5: 3,202 papers (Global 12,811 to 16,012) -> screening_groups/group_5_papers.csv


## 4. Helper Functions: Batch Generation & Direct API Screener


In [16]:
def create_batch_requests_file(group_records, output_file, model=DEFAULT_MODEL, reasoning_effort=DEFAULT_REASONING_EFFORT):
    """
    Builds an OpenAI Batch API compatible JSONL file.
    Uses /v1/chat/completions with JSON response format.
    Tailored for reasoning models (e.g. GPT-5.4 Thinking) and legacy chat models.
    """
    batch_lines = []
    is_reasoning_model = any(prefix in model.lower() for prefix in ["gpt-5", "o1", "o3"])

    for r in group_records:
        title = r["Title"]
        abstract = r["Abstract"] if r["Abstract"] else "[No abstract provided in bibliographic record]"
        prompt_content = f"Title: {title}\nAbstract: {abstract}"
        
        body = {
            "model": model,
            "response_format": {"type": "json_object"},
            "messages": [
                {"role": "system", "content": INSTRUCTIONS_RAW},
                {"role": "user", "content": prompt_content}
            ]
        }
        if is_reasoning_model:
            if reasoning_effort:
                body["reasoning_effort"] = reasoning_effort
        else:
            body["temperature"] = 0.0

        request_entry = {
            "custom_id": r["Study_ID"],
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": body
        }
        batch_lines.append(request_entry)
        
    with open(output_file, "w", encoding="utf-8") as f:
        for entry in batch_lines:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
            
    print(f"Generated {len(batch_lines):,} batch requests -> {output_file}")
    return output_file

def screen_single_study_direct(client, title, abstract, model=DEFAULT_MODEL, reasoning_effort=DEFAULT_REASONING_EFFORT, max_retries=3):
    """
    Screens a single study directly via OpenAI Chat Completions API with exponential backoff.
    Tailored for reasoning models (e.g. GPT-5.4 Thinking) and legacy chat models.
    """
    if not abstract:
        abstract = "[No abstract provided in bibliographic record]"
    prompt_content = f"Title: {title}\nAbstract: {abstract}"
    
    is_reasoning_model = any(prefix in model.lower() for prefix in ["gpt-5", "o1", "o3"])
    api_kwargs = {
        "model": model,
        "response_format": {"type": "json_object"},
        "messages": [
            {"role": "system", "content": INSTRUCTIONS_RAW},
            {"role": "user", "content": prompt_content}
        ],
        "timeout": 60.0
    }
    if is_reasoning_model:
        if reasoning_effort:
            api_kwargs["reasoning_effort"] = reasoning_effort
    else:
        api_kwargs["temperature"] = 0.0

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(**api_kwargs)
            raw_text = response.choices[0].message.content
            parsed = json.loads(raw_text)
            return {
                "Include?": parsed.get("Include?", "uncertain"),
                "Exclusion_reason": parsed.get("Exclusion_reason"),
                "Uncertain": parsed.get("Uncertain", "no"),
                "Comments": parsed.get("Comments", "")
            }
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                return {
                    "Include?": "error",
                    "Exclusion_reason": "API_ERROR",
                    "Uncertain": "yes",
                    "Comments": f"API request error: {str(e)}"
                }


## 5. GROUP 1 SCREENING (Execute First)
> "Take the first group of papers to do first
Prepare and execute the screening for **Group 1** (3,203 papers).

In [17]:
# Generate Batch Request File for Group 1
group1_records = group_partitions[1]["records"]
batch_group1_file = "batch_requests/bacteria_screening_batch_group_1.jsonl"

create_batch_requests_file(group1_records, batch_group1_file, model=DEFAULT_MODEL)


Generated 3,203 batch requests -> batch_requests/bacteria_screening_batch_group_1.jsonl


'batch_requests/bacteria_screening_batch_group_1.jsonl'

Direct API Screening for Group 1 (With Live Checkpointing)
This runner screens Group 1 directly with real-time progress, saving progress every 10 papers to `screening_results/group_1_direct_checkpoint.jsonl`.
If interrupted, it automatically resumes from the last screened paper


In [18]:
def run_group_direct_screening(group_records, group_num=1, model=DEFAULT_MODEL, reasoning_effort=DEFAULT_REASONING_EFFORT, sample_limit=None):
    """
    Runs direct screening on a group of papers with continuous auto-saving and checkpointing.
    Set sample_limit (e.g., 5 or 20) for a pilot test, or None for the full group.
    """
    if OpenAI is None or not OPENAI_API_KEY:
        print("OpenAI client or OPENAI_API_KEY is not configured.")
        print("Please set your OPENAI_API_KEY above to execute live screening.")
        return None
        
    client = OpenAI(api_key=OPENAI_API_KEY)
    checkpoint_file = f"screening_results/group_{group_num}_direct_checkpoint.jsonl"
    
    # Load existing progress if any
    screened_data = {}
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r", encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                screened_data[item["Study_ID"]] = item
        print(f"Resuming from checkpoint: {len(screened_data):,} papers already screened.")
        
    target_records = group_records[:sample_limit] if sample_limit else group_records
    
    with open(checkpoint_file, "a", encoding="utf-8") as f_out:
        for r in tqdm(target_records, desc=f"Screening Group {group_num}"):
            sid = r["Study_ID"]
            if sid in screened_data:
                continue
                
            decision = screen_single_study_direct(client, r["Title"], r["Abstract"], model=model, reasoning_effort=reasoning_effort)
            
            result_item = copy.deepcopy(r)
            result_item["LLM_Include"] = decision.get("Include?", "")
            result_item["LLM_Exclusion_Reason"] = decision.get("Exclusion_reason") or ""
            result_item["LLM_Uncertain"] = decision.get("Uncertain", "")
            result_item["LLM_Comments"] = decision.get("Comments", "")
            
            f_out.write(json.dumps(result_item, ensure_ascii=False) + "\n")
            f_out.flush()
            screened_data[sid] = result_item
            
    print(f"Group {group_num} direct screening complete: {len(screened_data):,} papers saved to {checkpoint_file}")
    return list(screened_data.values())

# PILOT TEST: Run first 5 papers of Group 1 to verify functionality (safe test)
# To run full Group 1, remove sample_limit or set sample_limit=None
# pilot_results = run_group_direct_screening(group1_records, group_num=1, sample_limit=5)


### Parse & Process Group 1 Results
Once Group 1 screening is complete (either via Batch API output or Direct API runner), merge decisions and export results:


In [19]:
def process_and_merge_group_results(group_num=1):
    """
    Merges LLM screening decisions with the original group papers,
    exporting both full screened CSV and an included studies subset.
    """
    checkpoint_file = f"screening_results/group_{group_num}_direct_checkpoint.jsonl"
    batch_output_file = f"screening_results/group_{group_num}_batch_output.jsonl"
    
    decisions_by_id = {}
    
    # 1. Check if Batch API results exist
    if os.path.exists(batch_output_file):
        with open(batch_output_file, "r", encoding="utf-8") as f:
            for line in f:
                res = json.loads(line)
                sid = res.get("custom_id")
                try:
                    content = res["response"]["body"]["choices"][0]["message"]["content"]
                    parsed = json.loads(content)
                    decisions_by_id[sid] = parsed
                except Exception:
                    pass
        print(f"Loaded {len(decisions_by_id):,} decisions from Batch API output.")
        
    # 2. Check direct screening checkpoint
    elif os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r", encoding="utf-8") as f:
            for line in f:
                res = json.loads(line)
                sid = res.get("Study_ID")
                decisions_by_id[sid] = {
                    "Include?": res.get("LLM_Include"),
                    "Exclusion_reason": res.get("LLM_Exclusion_Reason"),
                    "Uncertain": res.get("LLM_Uncertain"),
                    "Comments": res.get("LLM_Comments")
                }
        print(f"Loaded {len(decisions_by_id):,} decisions from Direct Screening checkpoint.")
    else:
        print(f"No screening output found for Group {group_num} yet.")
        return
        
    # Merge with group papers
    group_csv = f"screening_groups/group_{group_num}_papers.csv"
    output_screened_csv = f"screening_results/group_{group_num}_screened.csv"
    output_included_csv = f"screening_results/group_{group_num}_included_studies.csv"
    
    with open(group_csv, "r", encoding="utf-8-sig") as f_in:
        rows = list(csv.DictReader(f_in))
        
    included_count = 0
    excluded_count = 0
    uncertain_count = 0
    
    included_rows = []
    for r in rows:
        sid = r["Study_ID"]
        if sid in decisions_by_id:
            d = decisions_by_id[sid]
            inc = str(d.get("Include?", "")).strip().lower()
            r["LLM_Include"] = inc
            r["LLM_Exclusion_Reason"] = str(d.get("Exclusion_reason", "") or "")
            r["LLM_Uncertain"] = str(d.get("Uncertain", "") or "")
            r["LLM_Comments"] = str(d.get("Comments", "") or "")
            
            if inc == "yes":
                included_count += 1
                included_rows.append(r)
            else:
                excluded_count += 1
            if str(d.get("Uncertain", "")).lower() == "yes":
                uncertain_count += 1
                
    # Save full screened CSV
    with open(output_screened_csv, "w", encoding="utf-8-sig", newline="") as f_out:
        writer = csv.DictWriter(f_out, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
        
    # Save included studies CSV
    with open(output_included_csv, "w", encoding="utf-8-sig", newline="") as f_out:
        writer = csv.DictWriter(f_out, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(included_rows)
        
    print(f"\n=== Group {group_num} Screening Summary ===")
    print(f"Total Papers Screened : {len(decisions_by_id):,}")
    print(f"Included (Proceeding) : {included_count:,} ({included_count/len(decisions_by_id)*100:.1f}%)")
    print(f"Excluded              : {excluded_count:,} ({excluded_count/len(decisions_by_id)*100:.1f}%)")
    print(f"Uncertain Flagged     : {uncertain_count:,} ({uncertain_count/len(decisions_by_id)*100:.1f}%)")
    print(f"Full Results Saved    : {output_screened_csv}")
    print(f"Included Studies Saved: {output_included_csv}")

# Run merger if results exist
process_and_merge_group_results(group_num=1)


No screening output found for Group 1 yet.


These cells are prepared for Groups 2, 3, 4, and 5, but have been guarded with `RUN_GROUP_X = False`.


In [20]:
# GROUP 2 SCREENING - PENDING GROUP 1 EVALUATION & MANUAL AUDIT
RUN_GROUP_2 = False 

if not RUN_GROUP_2:
    print("Group 2 execution is LOCKED. (Set RUN_GROUP_2 = True when ready)")
else:
    print("Preparing Group 2 batch requests...")
    group2_records = group_partitions[2]["records"]
    batch_group2_file = "batch_requests/bacteria_screening_batch_group_2.jsonl"
    create_batch_requests_file(group2_records, batch_group2_file, model=DEFAULT_MODEL)
    # run_group_direct_screening(group2_records, group_num=2)


Group 2 execution is LOCKED. (Set RUN_GROUP_2 = True when ready)


In [21]:
# GROUP 3 SCREENING 
RUN_GROUP_3 = False 

if not RUN_GROUP_3:
    print("Group 3 execution is LOCKED. (Set RUN_GROUP_3 = True when ready)")
else:
    print("Preparing Group 3 batch requests...")
    group3_records = group_partitions[3]["records"]
    batch_group3_file = "batch_requests/bacteria_screening_batch_group_3.jsonl"
    create_batch_requests_file(group3_records, batch_group3_file, model=DEFAULT_MODEL)
    # run_group_direct_screening(group3_records, group_num=3)


Group 3 execution is LOCKED. (Set RUN_GROUP_3 = True when ready)


In [22]:
# GROUP 4 SCREENING
RUN_GROUP_4 = False 

if not RUN_GROUP_4:
    print("Group 4 execution is LOCKED. (Set RUN_GROUP_4 = True when ready)")
else:
    print("Preparing Group 4 batch requests...")
    group4_records = group_partitions[4]["records"]
    batch_group4_file = "batch_requests/bacteria_screening_batch_group_4.jsonl"
    create_batch_requests_file(group4_records, batch_group4_file, model=DEFAULT_MODEL)
    # run_group_direct_screening(group4_records, group_num=4)


Group 4 execution is LOCKED. (Set RUN_GROUP_4 = True when ready)


In [23]:
# GROUP 5 SCREENING 
RUN_GROUP_5 = False 

if not RUN_GROUP_5:
    print("Group 5 execution is LOCKED. (Set RUN_GROUP_5 = True when ready)")
else:
    print("Preparing Group 5 batch requests...")
    group5_records = group_partitions[5]["records"]
    batch_group5_file = "batch_requests/bacteria_screening_batch_group_5.jsonl"
    create_batch_requests_file(group5_records, batch_group5_file, model=DEFAULT_MODEL)
    # run_group_direct_screening(group5_records, group_num=5)


Group 5 execution is LOCKED. (Set RUN_GROUP_5 = True when ready)


In [24]:
RESULTS_DIR = Path("screening_results")
TOTAL_GROUPS = 5

print("=" * 65)
print("POST-SCREENING STATUS")
print("=" * 65)

group_stats = []
all_results = []
all_included = []

for g in range(1, TOTAL_GROUPS + 1):
    csv_path = RESULTS_DIR / f"group_{g}_screened.csv"
    jsonl_path = RESULTS_DIR / f"group_{g}_screened.jsonl"
    
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        total = len(df)
        decisions = df["screening_decision"].value_counts().to_dict()
        included = decisions.get("Include", 0)
        excluded = decisions.get("Exclude", 0)
        uncertain = decisions.get("Uncertain", 0)
        errors = df["screening_error"].notna().sum() if "screening_error" in df.columns else 0
        
        group_stats.append({
            "Group": f"Group {g}",
            "Status": "COMPLETED",
            "Screened": total,
            "Included": included,
            "Excluded": excluded,
            "Uncertain": uncertain,
            "Errors": errors
        })
        all_results.append(df)
        all_included.append(df[df["screening_decision"] == "Include"])
        
    elif jsonl_path.exists():
        # Fallback to JSONL if CSV not yet merged
        rows = []
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rows.append(json.loads(line))
        df = pd.DataFrame(rows)
        total = len(df)
        decisions = df["decision"].value_counts().to_dict()
        included = decisions.get("Include", 0)
        excluded = decisions.get("Exclude", 0)
        uncertain = decisions.get("Uncertain", 0)
        
        group_stats.append({
            "Group": f"Group {g}",
            "Status": "JSONL ONLY",
            "Screened": total,
            "Included": included,
            "Excluded": excluded,
            "Uncertain": uncertain,
            "Errors": 0
        })
    else:
        group_stats.append({
            "Group": f"Group {g}",
            "Status": "NOT SCREENED",
            "Screened": 0,
            "Included": 0,
            "Excluded": 0,
            "Uncertain": 0,
            "Errors": 0
        })

# 1. Group-by-Group Table
stats_df = pd.DataFrame(group_stats)
print("\n--- Group Status ---")
print(stats_df.to_string(index=False))

# 2. Overall Totals
total_screened = stats_df["Screened"].sum()
total_included = stats_df["Included"].sum()
total_excluded = stats_df["Excluded"].sum()
total_uncertain = stats_df["Uncertain"].sum()

print("\n" + "=" * 65)
print(f"Total Papers Screened : {total_screened:,}")
print(f"Papers Left (INCLUDED): {total_included:,}  ({total_included/max(total_screened,1)*100:.1f}%)")
print(f"Papers EXCLUDED       : {total_excluded:,}  ({total_excluded/max(total_screened,1)*100:.1f}%)")
print(f"Papers UNCERTAIN      : {total_uncertain:,}  ({total_uncertain/max(total_screened,1)*100:.1f}%)")
print("=" * 65)

# 3. Top Exclusion Reasons (if any papers have been screened)
if all_results:
    combined_df = pd.concat(all_results, ignore_index=True)
    if "exclusion_reasons" in combined_df.columns:
        excl_reasons = combined_df[combined_df["screening_decision"] == "Exclude"]["exclusion_reasons"].dropna()
        reason_counts = Counter()
        for r in excl_reasons:
            for item in str(r).split(";"):
                item = item.strip()
                if item:
                    reason_counts[item] += 1
        
        if reason_counts:
            print("\n--- Top Exclusion Reasons ---")
            for reason, count in reason_counts.most_common(10):
                print(f"  {reason:<35}: {count:>5} ({count/total_excluded*100:.1f}%)")


POST-SCREENING STATUS

--- Group Status ---
  Group       Status  Screened  Included  Excluded  Uncertain  Errors
Group 1 NOT SCREENED         0         0         0          0       0
Group 2 NOT SCREENED         0         0         0          0       0
Group 3 NOT SCREENED         0         0         0          0       0
Group 4 NOT SCREENED         0         0         0          0       0
Group 5 NOT SCREENED         0         0         0          0       0

Total Papers Screened : 0
Papers Left (INCLUDED): 0  (0.0%)
Papers EXCLUDED       : 0  (0.0%)
Papers UNCERTAIN      : 0  (0.0%)
